# 04 — Validation out-of-sample (TEST SET)
**Savvy Crypto Agent — Jedha Bootcamp Final Project**

C'est ici qu'on decouvre si le modele est reellement bon ou s'il a simplement memorise le TRAIN.

**Ce notebook est la seule utilisation legitime du test set. On ne retourne pas modifier le modele apres.**

## 0. Imports & config

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, joblib, warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import roc_auc_score, precision_recall_curve, roc_curve

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_palette('husl')

PROCESSED = "C:\\Users\\casir\\Desktop\\CRYPTOS\\data\\processed\\"
MODELS    = "C:\\Users\\casir\\Desktop\\CRYPTOS\\models\\"

with open(PROCESSED + 'config.json') as f:
    cfg = json.load(f)
with open(PROCESSED + 'ml_config.json') as f:
    ml  = json.load(f)

TP_NET       = cfg['tp_net']
SL_NET       = cfg['sl_net']
BREAKEVEN    = cfg['breakeven']
FEATURE_COLS = ml['feature_cols']
XGB_THRESH   = ml['xgb_threshold']
RF_THRESH    = ml['rf_threshold']
BEST_MODEL   = ml['best_model']

print(f'Breakeven     : {BREAKEVEN*100:.1f}%')
print(f'XGB seuil     : {XGB_THRESH}')
print(f'RF  seuil     : {RF_THRESH}')
print(f'Meilleur modele : {BEST_MODEL}')

## 1. Chargement du test set et des modeles

In [ ]:
test = pd.read_csv(PROCESSED + 'test_features.csv')
test['candle_time'] = pd.to_datetime(test['candle_time'], utc=True)
test = test.sort_values('candle_time').reset_index(drop=True)

X_test = test[FEATURE_COLS]
y_test = test['label']

xgb_model = joblib.load(MODELS + 'xgb_final.pkl')
rf_model  = joblib.load(MODELS + 'rf_final.pkl')

test['proba_xgb'] = xgb_model.predict_proba(X_test)[:, 1]
test['proba_rf']  = rf_model.predict_proba(X_test)[:, 1]

print(f'TEST : {len(test):,} bougies')
print(f'Periode : {test["candle_time"].min().date()} -> {test["candle_time"].max().date()}')
print(f'Win rate baseline TEST : {y_test.mean()*100:.1f}%')
print(f'XGB proba moy WIN  : {test[y_test==1]["proba_xgb"].mean():.3f}')
print(f'XGB proba moy LOSS : {test[y_test==0]["proba_xgb"].mean():.3f}')

## 2. AUC sur le test set

Rappel : sur le TRAIN l'AUC etait eleve. Si elle s'effondre sur le TEST -> surapprentissage.

In [ ]:
auc_xgb_test = roc_auc_score(y_test, test['proba_xgb'])
auc_rf_test  = roc_auc_score(y_test, test['proba_rf'])

print(f'AUC XGBoost      sur TEST : {auc_xgb_test:.4f}')
print(f'AUC Random Forest sur TEST : {auc_rf_test:.4f}')
print()
print('Interpretation :')
print('  AUC > 0.75 -> modele utile')
print('  AUC > 0.85 -> modele fort')
print('  AUC proche de 0.5 -> modele inutile (surapprentissage)')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for name, proba_col, color in [
    ('XGBoost', 'proba_xgb', 'steelblue'),
    ('Random Forest', 'proba_rf', 'coral')
]:
    fpr, tpr, _ = roc_curve(y_test, test[proba_col])
    auc = roc_auc_score(y_test, test[proba_col])
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)
    prec, rec, _ = precision_recall_curve(y_test, test[proba_col])
    axes[1].plot(rec, prec, label=name, color=color, linewidth=2)

axes[0].plot([0,1],[0,1],'k--', alpha=0.3, label='Aleatoire')
axes[0].set_title('Courbe ROC — TEST SET', fontsize=13)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

axes[1].axhline(BREAKEVEN, color='black', linestyle='--', label=f'Breakeven ({BREAKEVEN*100:.0f}%)')
axes[1].set_title('Precision-Recall — TEST SET', fontsize=13)
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision (win rate)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Validation des regles EDA sur le test set

Avant meme le ML : est-ce que les regles simples trouvees en EDA tiennent sur le test set ?

In [ ]:
train = pd.read_csv(PROCESSED + 'train_features.csv')

filters = [
    ('Stoch_K < 10',                        lambda d: d['stoch_k'] < 10),
    ('RSI<30 & Stoch<20',                   lambda d: (d['rsi14'] < 30) & (d['stoch_k'] < 20)),
    ('RSI<30 & Stoch<20 & PvsEMA50<-0.5%', lambda d: (d['rsi14'] < 30) & (d['stoch_k'] < 20) & (d['price_vs_ema50'] < -0.5)),
    ('RSI<35 & Stoch<20',                   lambda d: (d['rsi14'] < 35) & (d['stoch_k'] < 20)),
    ('RSI<35 & Stoch<20 & PvsEMA50<-0.5%', lambda d: (d['rsi14'] < 35) & (d['stoch_k'] < 20) & (d['price_vs_ema50'] < -0.5)),
    ('RSI<40 & Stoch<20',                   lambda d: (d['rsi14'] < 40) & (d['stoch_k'] < 20)),
]

print(f'Breakeven : {BREAKEVEN*100:.1f}%')
print()
print(f'{"Filtre":<45} {"TRAIN wr":>9} {"TRAIN n":>8} {"TEST wr":>9} {"TEST n":>8} {"Signal":>10}')
print('-' * 100)

for label, fn in filters:
    tr_mask = fn(train)
    te_mask = fn(test)
    tr_wr = train[tr_mask]['label'].mean() * 100 if tr_mask.sum() > 0 else 0
    te_wr = test[te_mask]['label'].mean() * 100   if te_mask.sum() > 0 else 0
    signal = 'OK TIENT' if te_wr > BREAKEVEN * 100 else ('PROCHE' if te_wr > BREAKEVEN * 100 * 0.85 else 'CASSE')
    print(f'{label:<45} {tr_wr:>8.1f}% {tr_mask.sum():>8,} {te_wr:>8.1f}% {te_mask.sum():>8,} {signal:>10}')

## 4. Analyse du seuil de confiance sur le test set

In [ ]:
def threshold_analysis_test(proba_col, model_name, color, optimal_thresh):
    thresholds = np.arange(0.05, 0.98, 0.01)
    results = []
    for t in thresholds:
        mask = test[proba_col] >= t
        sub  = test[mask]
        if len(sub) < 30:
            break
        wr  = sub['label'].mean()
        n   = len(sub)
        ev  = wr * TP_NET + (1 - wr) * (-SL_NET)
        results.append({
            'threshold': round(t, 2),
            'n_trades' : n,
            'win_rate' : round(wr * 100, 2),
            'ev_pct'   : round(ev * 100, 3),
            'rentable' : wr > BREAKEVEN
        })
    df = pd.DataFrame(results)

    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
    fig.suptitle(f'Seuil de confiance sur TEST SET — {model_name}', fontsize=13)

    colors = ['steelblue' if r else 'tomato' for r in df['rentable']]
    axes[0].bar(df['threshold'], df['win_rate'], color=colors, width=0.01)
    axes[0].axhline(BREAKEVEN*100, color='black', linestyle='--', label=f'Breakeven ({BREAKEVEN*100:.0f}%)')
    axes[0].axvline(optimal_thresh, color='green', linestyle='--', linewidth=2, label=f'Seuil TRAIN optimal ({optimal_thresh})')
    axes[0].set_ylabel('Win rate (%)')
    axes[0].legend()

    axes[1].bar(df['threshold'], df['n_trades'], color=color, alpha=0.6, width=0.01)
    axes[1].axvline(optimal_thresh, color='green', linestyle='--', linewidth=2)
    axes[1].set_ylabel('Nombre de trades')

    colors2 = ['steelblue' if e > 0 else 'tomato' for e in df['ev_pct']]
    axes[2].bar(df['threshold'], df['ev_pct'], color=colors2, width=0.01)
    axes[2].axhline(0, color='black', linewidth=0.8)
    axes[2].axvline(optimal_thresh, color='green', linestyle='--', linewidth=2)
    axes[2].set_ylabel('Expected value / trade (%)')
    axes[2].set_xlabel('Seuil de probabilite')

    plt.tight_layout()
    plt.show()

    # Stats au seuil optimal du TRAIN
    at_optimal = df[df['threshold'] == optimal_thresh]
    if len(at_optimal) > 0:
        row = at_optimal.iloc[0]
        print(f'=== AU SEUIL TRAIN OPTIMAL ({optimal_thresh}) — {model_name} sur TEST ===')
        print(f'  Win rate  : {row["win_rate"]}%  (seuil rentabilite : {BREAKEVEN*100:.0f}%)')
        print(f'  N trades  : {row["n_trades"]:,}')
        print(f'  EV/trade  : {row["ev_pct"]}%')
        print(f'  Rentable  : {row["rentable"]}')

    rentable = df[(df['rentable']) & (df['n_trades'] > 50)]
    if len(rentable) > 0:
        best = rentable.loc[rentable['ev_pct'].idxmax()]
        print(f'\nMeilleur seuil sur TEST :')
        print(f'  Seuil     : {best["threshold"]}')
        print(f'  Win rate  : {best["win_rate"]}%')
        print(f'  N trades  : {best["n_trades"]:,}')
        print(f'  EV/trade  : {best["ev_pct"]}%')

    return df

df_xgb_test = threshold_analysis_test('proba_xgb', 'XGBoost', 'steelblue', XGB_THRESH)

In [ ]:
df_rf_test = threshold_analysis_test('proba_rf', 'Random Forest', 'coral', RF_THRESH)

## 5. Backtest complet sur le test set

In [ ]:
def backtest_test(model_name, proba_col, thresh):
    if thresh is None:
        print(f'Pas de seuil pour {model_name}')
        return

    trades = test[test[proba_col] >= thresh].copy().sort_values('candle_time')
    CAPITAL = 10_000
    SIZE    = 500

    trades['pnl']     = trades['exit_ret'] * SIZE
    trades['capital'] = CAPITAL + trades['pnl'].cumsum()
    trades['dd']      = trades['capital'] / trades['capital'].cummax() - 1

    total = len(trades)
    wins  = trades['label'].sum()
    pnl   = trades['pnl'].sum()
    final = CAPITAL + pnl
    days  = (trades['candle_time'].max() - trades['candle_time'].min()).days
    tpd   = total / max(days, 1)
    ppd   = pnl   / max(days, 1)

    print(f'=== BACKTEST TEST — {model_name} (seuil={thresh}) ===')
    print(f'Periode         : {trades["candle_time"].min().date()} -> {trades["candle_time"].max().date()}')
    print(f'Total trades    : {total:,}  |  Win rate : {wins/total*100:.1f}%')
    print(f'PnL total       : {pnl:+,.0f} EUR')
    print(f'Capital final   : {final:,.0f} EUR  ({pnl/CAPITAL*100:+.1f}%)')
    print(f'Max drawdown    : {trades["dd"].min()*100:.1f}%')
    print(f'Trades/jour     : {tpd:.1f}')
    print(f'PnL/jour        : {ppd:+.1f} EUR  (objectif : {CAPITAL*0.005:.0f} EUR = 0.5%/jour)')
    obj = 'ATTEINT' if ppd >= CAPITAL * 0.005 else 'NON ATTEINT'
    print(f'Objectif 0.5%/j : {obj}')
    print()

    # Breakdown par symbole
    print('Par symbole :')
    sym_stats = trades.groupby('symbol').agg(
        n=('label','count'),
        wr=('label','mean'),
        pnl=('pnl','sum')
    ).sort_values('pnl', ascending=False)
    sym_stats['wr_pct'] = sym_stats['wr'] * 100
    print(sym_stats[['n','wr_pct','pnl']].to_string())

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    ax1.plot(trades['candle_time'], trades['capital'], color='steelblue', linewidth=1.5)
    ax1.axhline(CAPITAL, color='gray', linestyle=':')
    ax1.fill_between(trades['candle_time'], CAPITAL, trades['capital'],
                     where=trades['capital'] >= CAPITAL, alpha=0.2, color='steelblue')
    ax1.fill_between(trades['candle_time'], CAPITAL, trades['capital'],
                     where=trades['capital'] < CAPITAL, alpha=0.2, color='tomato')
    ax1.set_title(f'Capital — {model_name} seuil={thresh} (TEST SET)', fontsize=13)
    ax1.set_ylabel('Capital (EUR)')

    ax2.fill_between(trades['candle_time'], trades['dd']*100, 0, color='tomato', alpha=0.4)
    ax2.set_title('Drawdown (%)')
    ax2.set_ylabel('Drawdown (%)')
    ax2.set_xlabel('Date')
    plt.tight_layout()
    plt.show()

backtest_test('XGBoost', 'proba_xgb', XGB_THRESH)
backtest_test('Random Forest', 'proba_rf', RF_THRESH)

## 6. Analyse mensuelle — stabilite du signal dans le temps

In [ ]:
best_proba = 'proba_xgb' if BEST_MODEL == 'XGBoost' else 'proba_rf'
best_thresh = XGB_THRESH if BEST_MODEL == 'XGBoost' else RF_THRESH

trades_best = test[test[best_proba] >= best_thresh].copy()
trades_best['month'] = trades_best['candle_time'].dt.to_period('M')

monthly = trades_best.groupby('month').agg(
    n=('label','count'),
    wr=('label','mean'),
    pnl=('exit_ret', lambda x: (x * 500).sum())
).reset_index()
monthly['month_str'] = monthly['month'].astype(str)
monthly['wr_pct'] = monthly['wr'] * 100

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
fig.suptitle(f'Analyse mensuelle — {BEST_MODEL} seuil={best_thresh} (TEST SET)', fontsize=13)

colors_wr = ['steelblue' if w > BREAKEVEN*100 else 'tomato' for w in monthly['wr_pct']]
ax1.bar(monthly['month_str'], monthly['wr_pct'], color=colors_wr)
ax1.axhline(BREAKEVEN*100, color='black', linestyle='--', label=f'Breakeven ({BREAKEVEN*100:.0f}%)')
ax1.set_ylabel('Win rate (%)')
ax1.legend()
for i, (_, row) in enumerate(monthly.iterrows()):
    ax1.text(i, row['wr_pct']+0.5, f'{row["wr_pct"]:.1f}%', ha='center', fontsize=9)

ax2.bar(monthly['month_str'], monthly['n'], color='steelblue', alpha=0.7)
ax2.set_ylabel('Nombre de trades')

colors_pnl = ['steelblue' if p > 0 else 'tomato' for p in monthly['pnl']]
ax3.bar(monthly['month_str'], monthly['pnl'], color=colors_pnl)
ax3.axhline(0, color='black', linewidth=0.8)
ax3.set_ylabel('PnL mensuel (EUR)')
ax3.set_xlabel('Mois')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Resultats mensuels :')
print(monthly[['month_str','n','wr_pct','pnl']].to_string(index=False))

## 7. Synthese finale

In [ ]:
best_proba  = 'proba_xgb' if BEST_MODEL == 'XGBoost' else 'proba_rf'
best_thresh = XGB_THRESH  if BEST_MODEL == 'XGBoost' else RF_THRESH
trades_best = test[test[best_proba] >= best_thresh].copy()
trades_best['pnl'] = trades_best['exit_ret'] * 500
total  = len(trades_best)
wins   = trades_best['label'].sum()
pnl    = trades_best['pnl'].sum()
days   = (trades_best['candle_time'].max() - trades_best['candle_time'].min()).days

print('=' * 60)
print('SYNTHESE FINALE — VALIDATION OUT-OF-SAMPLE')
print('=' * 60)
print()
print(f'Modele          : {BEST_MODEL}')
print(f'Seuil           : {best_thresh}')
print(f'Periode TEST    : {test["candle_time"].min().date()} -> {test["candle_time"].max().date()}')
print(f'Breakeven       : {BREAKEVEN*100:.1f}%')
print()
print(f'Win rate        : {wins/total*100:.1f}%  ({'AU-DESSUS' if wins/total > BREAKEVEN else 'EN-DESSOUS'} du breakeven)')
print(f'Total trades    : {total:,}')
print(f'PnL total       : {pnl:+,.0f} EUR')
print(f'Capital 10k EUR : {10000+pnl:,.0f} EUR  ({pnl/10000*100:+.1f}%)')
print(f'PnL / jour      : {pnl/max(days,1):+.1f} EUR  (objectif : 50 EUR)')
print(f'Objectif 0.5%/j : {"ATTEINT" if pnl/max(days,1) >= 50 else "NON ATTEINT"}')
print()
print('CONCLUSION :')
if wins/total > BREAKEVEN and pnl > 0:
    print('  Le modele est rentable sur le test set out-of-sample.')
    print('  Le signal identifie en EDA (mean reversion sur survente) est reel.')
    print('  Prochaine etape : deploiement en paper trading pour validation live.')
else:
    print('  Le modele ne tient pas sur le test set.')
    print('  Pistes : ajuster le seuil, enrichir les features, revoir la labellisation.')
print('=' * 60)

In [ ]:
import json
from datetime import datetime

best_proba  = 'proba_xgb' if BEST_MODEL == 'XGBoost' else 'proba_rf'
best_thresh = XGB_THRESH  if BEST_MODEL == 'XGBoost' else RF_THRESH
trades_best = test[test[best_proba] >= best_thresh].copy()
trades_best['pnl'] = trades_best['exit_ret'] * 500
total  = len(trades_best)
wins   = trades_best['label'].sum()
pnl    = trades_best['pnl'].sum()
days   = (trades_best['candle_time'].max() - trades_best['candle_time'].min()).days

recap = {
    'date_generated'    : datetime.now().isoformat(),
    'model'             : BEST_MODEL,
    'threshold'         : best_thresh,
    'tp_pct'            : cfg['tp_pct'],
    'sl_pct'            : cfg['sl_pct'],
    'fees_pct'          : cfg['fees_pct'],
    'breakeven'         : BREAKEVEN,
    'test_period_start' : str(test['candle_time'].min().date()),
    'test_period_end'   : str(test['candle_time'].max().date()),
    'total_trades'      : int(total),
    'win_trades'        : int(wins),
    'win_rate_pct'      : round(wins / total * 100, 2),
    'pnl_total_eur'     : round(pnl, 2),
    'capital_initial'   : 10000,
    'capital_final'     : round(10000 + pnl, 2),
    'return_pct'        : round(pnl / 10000 * 100, 2),
    'max_drawdown_pct'  : round(trades_best.assign(
                            capital=10000 + trades_best['pnl'].cumsum()
                          ).pipe(lambda d: (d['capital'] / d['capital'].cummax() - 1)).min() * 100, 2),
    'trades_per_day'    : round(total / max(days, 1), 1),
    'pnl_per_day_eur'   : round(pnl / max(days, 1), 2),
    'objective_05pct_j' : bool(pnl / max(days, 1) >= 50),
    'per_symbol'        : {
                        sym: {
                            'n'       : int(g['label'].count()),
                            'wr_pct'  : round(float(g['label'].mean()) * 100, 2),
                            'pnl_eur' : round(float(g['pnl'].sum()), 2)
                        }
                        for sym, g in trades_best.groupby('symbol')
                      }
}

with open(PROCESSED + 'final_recap.json', 'w') as f:
    json.dump(recap, f, indent=2)

print(f'Recap sauvegarde : {PROCESSED}final_recap.json')
print(json.dumps({k: v for k, v in recap.items() if k != 'per_symbol'}, indent=2))